[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/Week%203/claude_code_example/sentiment_classifier_colab.ipynb)

# Movie Review Sentiment Classifier

TF-IDF + logistic regression on the `rotten_tomatoes` dataset — the notebook version of the Claude Code demo project. Runs top to bottom in Colab; no Hugging Face download, so nothing to rate-limit or fail.

## 1. Setup

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import pickle
print('ready')

## 2. Train

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/Week%203/rotten_tomatoes.csv"

try:
    df = pd.read_csv(DATA_URL)            # primary: read straight from GitHub
except Exception:
    df = pd.read_csv("rotten_tomatoes.csv")  # offline fallback: upload to Colab

train = df[df['split'] == 'train']
test  = df[df['split'] == 'test']

vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X_train = vectorizer.fit_transform(train['text'])
X_test  = vectorizer.transform(test['text'])

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, train['label'])

preds = clf.predict(X_test)
print(f"test accuracy: {accuracy_score(test['label'], preds):.4f}\n")
print(classification_report(test['label'], preds, target_names=['negative', 'positive']))

with open("model.pkl", "wb") as f:
    pickle.dump({"vectorizer": vectorizer, "classifier": clf}, f)
print("saved model.pkl")

## 3. Predict

In [ ]:
LABELS = {0: 'negative', 1: 'positive'}

def predict(text):
    X = vectorizer.transform([text])
    return LABELS[int(clf.predict(X)[0])]

for s in [
    'a sharp, funny movie that earns its ending',
    'a dull, lifeless waste of two hours',
    'this is not a good film',   # note: the model gets this one wrong
]:
    print(f"{predict(s):>8}  <-  {s}")

## 4. Unit test — green vs. red

A unit test for the `predict` function, written two ways. Run **one** of them.

- **GREEN** asserts the model's *actual* answers — including the negation case, which it documents as a known blind spot. Everything passes.
- **RED** asserts what a *human* knows is correct for that same sentence. The model fails it, raising an `AssertionError`: the test is right, the model is wrong. That gap is the whole point — generate fast, verify always.

In [ ]:
# GREEN  -- run this and nothing errors.
# The negation case asserts the model's real (wrong) answer, documenting the
# blind spot instead of pretending it's fixed.
assert predict('a brilliant, moving, beautifully acted film') == 'positive'
assert predict('a dull, lifeless waste of two hours') == 'negative'
assert predict('this is not a good film') == 'positive'   # known blind spot
print('all green - 3/3 passed')

**Red version** (run to see it fail):

In [ ]:
# RED  -- assert what a human KNOWS is correct. The model fails it.
# Running this raises AssertionError: the test is right, the model is wrong.
assert predict('this is not a good film') == 'negative', \
    "model predicted positive -- bag-of-words can't see the 'not'"